# Modelamiento ATU — índice (un modelo por sistema)

**Decisión de diseño:** no un modelo Lima-global. Un notebook / fit por tipo de transporte
(`metro_l1`, `troncal`, `corredor`, `alimentador`).

Cada notebook corre el bakeoff Chronos / LightGBM / AutoETS + baselines + aforo proxy +
regla U + anomalías, y escribe en `outputs/<sistema>/`.

Este índice solo **consolida WMAPE** para ver el ganador por modo.


## Comparativa WMAPE — ¿qué vamos a hacer?

Leer `metricas_modelos.csv` de cada sistema y armar una tabla pivote (filas = sistema,
columnas = modelo) con WMAPE y MAE. El ganador por fila es el menor WMAPE.


In [1]:

from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / "outputs").exists():
    RAIZ = RAIZ / "code" / "model" if (RAIZ / "code" / "model").exists() else RAIZ
# locate model dir
for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (cand / "code" / "model" / "outputs").exists():
        MOD = cand / "code" / "model"
        break
    if (cand / "outputs" / "metro_l1").exists():
        MOD = cand
        break
else:
    raise FileNotFoundError("outputs no encontrados")

rows = []
for sis in ["metro_l1", "troncal", "corredor", "alimentador"]:
    p = MOD / "outputs" / sis / "metricas_modelos.csv"
    if not p.exists():
        continue
    m = pd.read_csv(p, index_col=0)
    for modelo, r in m.iterrows():
        rows.append({"sistema": sis, "modelo": modelo, **r.to_dict()})
comp = pd.DataFrame(rows)
print(comp.pivot_table(index="sistema", columns="modelo", values="WMAPE").round(3).to_string())
print()
print(comp.pivot_table(index="sistema", columns="modelo", values="MAE").round(1).to_string())


modelo       autoets  chronos  hora_media  lightgbm  naive_7d
sistema                                                      
alimentador    0.483    0.507       0.624     0.529     0.658
corredor       0.215    0.311       0.353     0.347     0.410
metro_l1       0.179    0.155       0.198     0.161     0.193
troncal        0.285    0.310       0.329     0.350     0.412

modelo       autoets  chronos  hora_media  lightgbm  naive_7d
sistema                                                      
alimentador     40.3     37.9        46.2      40.1      49.5
corredor       156.6    173.9       234.9     199.2     239.8
metro_l1       159.9    131.5       175.6     138.0     165.1
troncal         85.9     74.0        94.9      87.3     103.1


### Qué encontramos (comparativa)

| Sistema | Ganador WMAPE | Lectura |
|---------|---------------|---------|
| Metro L1 | **Chronos** (~0.155) | Panel limpio; foundation gana |
| Troncal | **AutoETS** (~0.285) | Clásico estacional supera a Chronos en relativo |
| Corredores | **AutoETS** (~0.215) | Mayor margen vs Chronos |
| Alimentadores | **AutoETS** (~0.483) | Sigue siendo el modo más difícil |

**A qué puede deberse:** L1 tiene cobertura completa y calendario estable; buses tienen
más ruido, hubs y (en alimentadores) ceros estructurales.

**Decisión:**
1. Separar modelos por sistema (confirmado).
2. Default 24h = ganador local (no forzar Chronos en todos).
3. Alimentadores solo como piloto; no mezclar su WMAPE con L1 en un score único.
